# День 7 — Итоговый мини-проект

## Цель
Собрать **одну аккуратную работу** за неделю: открыл репозиторий → понял задачу → увидел baseline, модели, метрики и выводы.

Цепочка решения:

**задача → данные → EDA → split → baseline → 2 модели → метрики → выводы**

Датасет: **Diabetes** (регрессия) — продолжение days 5–6. Дерево берём с `max_depth=2` (лучший CV из day 6).

## 1. Задача

| | |
|---|---|
| **Тип** | регрессия |
| **Предсказываем** | прогрессию диабета за год (`y`, непрерывное число) |
| **По признакам** | 10 клинических показателей (age, bmi, bp, …) |
| **Успех** | модели бьют baseline; сравниваем MAE / RMSE / R² |

## 2. Данные + короткий EDA

In [ ]:
from sklearn.datasets import load_diabetes

dataset = load_diabetes(as_frame=True)
X = dataset.data
y = dataset.target

print("shape X:", X.shape)
print("признаки:", list(X.columns))
print("пропуски в X:", int(X.isna().sum().sum()))
print("пропуски в y:", int(y.isna().sum()))
print(f"y: min={y.min():.1f}, max={y.max():.1f}, mean={y.mean():.1f}, std={y.std():.1f}")
X.describe().round(3)

## 3. Train / test split

Тот же split, что в days 5–6: `test_size=0.2`, `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 4. Baseline

`DummyRegressor(strategy="mean")` — всегда среднее `y_train`. Планка, которую обязаны побить.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }


baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
m_baseline = regression_metrics(y_test, baseline.predict(X_test))

print(f"Baseline MAE:  {m_baseline['MAE']:.2f}")
print(f"Baseline RMSE: {m_baseline['RMSE']:.2f}")
print(f"Baseline R²:   {m_baseline['R2']:.3f}")

## 5. Две модели

1. **LinearRegression** — лучшая модель day 5.
2. **DecisionTreeRegressor(`max_depth=2`)** — лучший CV из day 6 (не неограниченное дерево).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
m_lr = regression_metrics(y_test, y_pred_lr)

tree = DecisionTreeRegressor(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
m_tree = regression_metrics(y_test, y_pred_tree)

print("LinearRegression:", {k: round(v, 3) for k, v in m_lr.items()})
print("DecisionTree max_depth=2:", {k: round(v, 3) for k, v in m_tree.items()})

## 6. Метрики — сравнение

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    [
        {"model": "Baseline (DummyRegressor mean)", **m_baseline},
        {"model": "LinearRegression", **m_lr},
        {"model": "DecisionTreeRegressor (max_depth=2)", **m_tree},
    ]
).round(3)

comparison = comparison.sort_values("MAE").reset_index(drop=True)
comparison

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

figures_dir = Path("week-3/figures")
if not figures_dir.exists():
    figures_dir = Path("../figures")
figures_dir.mkdir(parents=True, exist_ok=True)

fig_path = figures_dir / "07_y_true_vs_y_pred_linear_regression.png"

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.7, edgecolor="k")
lo = min(y_test.min(), y_pred_lr.min())
hi = max(y_test.max(), y_pred_lr.max())
plt.plot([lo, hi], [lo, hi], "r--", linewidth=2)
plt.title("Day 7: y_true vs y_pred (LinearRegression)")
plt.xlabel("y_true")
plt.ylabel("y_pred")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.show()

print(f"График: {fig_path}")

## 7. Выводы

1. **Задача:** регрессия на Diabetes — предсказать прогрессию диабета по 10 признакам (442 × 10, без пропусков).
2. **Baseline** (среднее): MAE ≈ 64, R² ≈ 0 — нижняя планка.
3. **LinearRegression** лучшая: MAE ≈ 42.8, R² ≈ 0.45 — заметно бьёт baseline (~21 по MAE).
4. **Дерево `max_depth=2`** лучше baseline и лучше неограниченного дерева из day 5, но слабее линейной модели на этом test-split.
5. Ошибка всё ещё существенная (MAE ~⅓ типичного `y`) — простые модели объясняют часть сигнала, не всё.
6. За неделю: Git → постановка задачи → EDA/baseline → классификация (Iris) → регрессия → CV/overfitting → **собранный мини-проект**.
7. Дальше: больше моделей (Ridge / RandomForest), feature engineering, отдельный hold-out / повторный CV.

Отчёт: `week-3/reports/week03_summary.md` · метрики также в README.

---

# Day 7 — Final mini-project

## Goal
One clean piece of work: **task → data → EDA → split → baseline → 2 models → metrics → conclusions**.

Dataset: **Diabetes** (regression). Tree uses `max_depth=2` from day 6 CV.

## Conclusions

1. Regression on Diabetes: predict disease progression from 10 features.
2. Baseline MAE ≈ 64 / R² ≈ 0; **LinearRegression** wins (MAE ≈ 42.8, R² ≈ 0.45).
3. Constrained tree (`max_depth=2`) beats baseline and the unrestricted tree from day 5, but loses to linear here.
4. Week takeaway: honest split + baseline + CV before trusting a model.
5. Next: stronger models / features; keep checking with CV.

Report: `week-3/reports/week03_summary.md` · metrics also in README.